# Create table with jumps from all mutations and sites
final dataframe will be saved in `outputs/jumps.parquet`
### In order to create jump dataframe:
1. run graph creation for each site
2. filter nodes that are not jumping 
3. add information about mutation
4. concatenate and save dataframes

In [32]:
import sys, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
DATA_PATH    = PROJECT_ROOT / 'data' / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH    = PROJECT_ROOT / 'data' / '01-readme-experiment-description_2022-04-05.csv'
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'
FINAL_OUTPUT_PATH = PROJECT_ROOT / 'outputs'
EXP_ID = 1
SIGNAL_COL = 'ERKKTR_ratio'
MAX_CPU_WORKERS = 8

In [33]:
meta_df = pd.read_csv(META_PATH)

In [34]:
def run_site(site, mutation=None):
    print(f"Starting graph computation for site {site}...")
    cmd = [
        sys.executable,
        str(SCRIPTS_DIR / 'spatiotemporal_signal_propagation.py'),
        '--data-path', str(DATA_PATH),
        '--meta-path', str(META_PATH),
        '--exp-id', str(EXP_ID),
        '--site-id', str(site),
        '--signal-col', SIGNAL_COL,
        '--output-dir', str(OUTPUT_ROOT),
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return site, res

futures = []
with ThreadPoolExecutor(max_workers=MAX_CPU_WORKERS) as ex:
    for site, mutation in zip(meta_df['Site'], meta_df['Mutation']):
        futures.append(ex.submit(run_site, site, mutation))

for fut in as_completed(futures):
    site_id, res = fut.result()
    print(f"--- Output for site {site_id} ---")
    print(res.stdout)
    if res.returncode != 0:
        print(f"Error for site {site_id}:\n", res.stderr)

Starting graph computation for site 1...Starting graph computation for site 2...

Starting graph computation for site 3...
Starting graph computation for site 4...
Starting graph computation for site 5...
Starting graph computation for site 6...
Starting graph computation for site 7...
Starting graph computation for site 8...
Starting graph computation for site 9...
Starting graph computation for site 10...
Starting graph computation for site 11...
Starting graph computation for site 12...
Starting graph computation for site 13...
Starting graph computation for site 14...
Starting graph computation for site 15...
Starting graph computation for site 16...
Starting graph computation for site 17...
Starting graph computation for site 18...
Starting graph computation for site 19...
Starting graph computation for site 20...
Starting graph computation for site 21...
Starting graph computation for site 22...
Starting graph computation for site 23...
Starting graph computation for site 24...
S

## Join dataframes

In [35]:
partial_dfs = []
for site, mutation in zip(meta_df['Site'], meta_df['Mutation']):
    try:
        df = pd.read_csv(OUTPUT_ROOT / f"exp_{EXP_ID}_site_{site}_ERKKTR_ratio/nodes.csv.gz")
    except FileNotFoundError:
        continue

    df['Site'] = site
    df['Mutation'] = mutation
    df = df[df['jump_event'] == True]  # noqa: E712
    partial_dfs.append(df)
full_df = pd.concat(partial_dfs)

In [38]:
full_df.to_parquet(FINAL_OUTPUT_PATH / "jumps.parquet")

In [39]:
full_df

,node_id,Exp_ID,Image_Metadata_Site,track_id,Image_Metadata_T,time_h,objNuclei_Location_Center_X,objNuclei_Location_Center_Y,Nuclear_size,ERKKTR_ratio,...,signal_value,signal_delta,jump_event,neighbor_count,neighbor_jump_count,neighbor_jump_now,neighbor_mean_signal,future_self_jump,Site,Mutation
1,1,1,1,1,1,0.083333,932.150,874.17400,333.0000,0.848242,...,0.848242,0.143835,True,22,7,True,0.963904,True,1,WT
2,2,1,1,1,2,0.166667,932.376,873.78700,314.0000,1.059170,...,1.059170,0.210928,True,22,5,True,1.004497,True,1,WT
3,3,1,1,1,3,0.250000,932.168,873.45300,322.0000,1.188000,...,1.188000,0.128830,True,22,8,True,1.029900,False,1,WT
13,13,1,1,1,13,1.083333,931.003,871.11700,309.0000,0.886782,...,0.886782,0.051232,True,21,2,True,0.899042,False,1,WT
20,20,1,1,1,20,1.666667,929.783,869.18400,309.0000,0.736695,...,0.736695,0.039816,True,19,1,True,0.833145,False,1,WT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251809,251809,1,20,1629,240,20.000000,329.773,217.60000,75.0001,0.994929,...,0.994929,0.150480,True,17,2,True,0.927403,True,20,PTEN_del
251810,251810,1,20,1629,241,20.083333,330.843,217.28600,70.0001,1.075350,...,1.075350,0.080421,True,17,1,True,0.934849,False,20,PTEN_del
251829,251829,1,20,1634,242,20.166667,1018.450,358.53200,154.0000,1.419850,...,1.419850,0.191050,True,6,0,False,0.929459,False,20,PTEN_del
251935,251935,1,20,1641,249,20.750000,103.512,2.70543,129.0000,1.208550,...,1.208550,0.069210,True,5,0,False,0.938208,True,20,PTEN_del
